位于 HTTP 请求和具体路由之间，能够对所有或一组请求进行通用处理.
```
客户端
  ↓
Middleware 请求阶段
  ↓
FastAPI 路由、依赖和业务逻辑
  ↓
Middleware 响应阶段
  ↓
客户端
```

In [ ]:
# 最基本的middleware
import time

from fastapi import FastAPI, Request

app = FastAPI()


@app.middleware("http")
async def add_process_time(
    request: Request,
    call_next,
):
    start_time = time.perf_counter()

    response = await call_next(request)

    process_time = time.perf_counter() - start_time
    response.headers["X-Process-Time"] = str(
        process_time
    )

    return response

#### 接受2个参数
##### request
当前 HTTP 请求对象，可以读取：
- request.method
- request.url
- request.headers
- request.client
- request.state

##### call_next
把请求继续交给**后面的 Middleware** 或 **FastAPI 路由**处理
`response = await call_next(request)`

执行完成后返回 Response。

如果不执行 call_next()，请求就不会继续进入路由。

### 洋葱模型
存在多个 Middleware 时，它们会一层一层包裹应用

In [ ]:
app.add_middleware(MiddlewareA)
app.add_middleware(MiddlewareB)

后添加的 MiddlewareB 位于外层

请求进入：
1. B before
2. A before
3. Route
   
响应返回：·
1. Route
2. B after
3. A after

请求从外向内，响应从内向外。

### Request ID
分布式后端经常需要给每个请求生成唯一标识

In [ ]:
from uuid import uuid4


@app.middleware("http")
async def request_id_middleware(
    request: Request,
    call_next,
):
    request_id = request.headers.get(
        "X-Request-ID"
    )

    if request_id is None:
        request_id = str(uuid4())

    request.state.request_id = request_id

    response = await call_next(request)

    response.headers["X-Request-ID"] = request_id
    return response

作用包括：
- 在日志中关联同一个请求
- 跨服务传递调用链标识
- 用户报告错误时帮助查询日志
- 将请求日志和异常日志关联起来

### 提前返回响应
Middleware 可以不执行`await call_next(request)`,而是提前返回：

In [ ]:
from fastapi.responses import JSONResponse


@app.middleware("http")
async def maintenance_middleware(
    request: Request,
    call_next,
):
    maintenance_mode = True

    if maintenance_mode:
        return JSONResponse(
            status_code=503,
            content={
                "code": "SERVICE_UNAVAILABLE",
                "message": "Maintenance in progress",
            },
        )

    return await call_next(request)

# 后续 Middleware 不执行
# 依赖不执行
# 路由函数不执行

适合：
- 系统维护模式
- 全局限流
- 非法 Host 拦截
- 特定网络层安全检查

### 修改响应
Middleware 可以修改通用响应头
```
@app.middleware("http")
async def security_headers(
    request: Request,
    call_next,
):
    response = await call_next(request)

    response.headers[
        "X-Content-Type-Options"
    ] = "nosniff"

    response.headers[
        "X-Frame-Options"
    ] = "DENY"

    return response
```
适合：
- 安全响应头
- Request ID
- 处理耗时
- 缓存策略
- 通用追踪字段
  
如果浏览器前端需要读取自定义响应头，还需要在 CORS 配置中公开相应 Header。

### Middleware 和 Dependency 的区别
#### Middleware
围绕 HTTP 请求工作：请求前 → 整个应用 → 响应后
特点：
- 通常作用于所有请求
- 能读取和修改 Request/Response
- 适合横切能力
- 不直接向路由参数注入业务对象

典型用途：
- 日志
- 耗时
- Request ID
- CORS
- 安全响应头
- 全局限流

#### Dependency
围绕业务对象工作：解析依赖 → 注入结果 → 路由使用
特点：
- 可以只用于指定路由
- 可以把值注入函数参数
- 可以形成依赖树
- 可以使用 yield 管理资源
- 方便测试替换

典型用途：
- 数据库连接
- 缓存
- 第三方 API 调用
- 文件上传


与 HTTP 请求整体相关，用 Middleware；路由需要某个对象或前置条件，用 Dependency。

### Middleware 和 Exception Handler 的区别
它们都可能处理异常，但职责不同。

#### Exception Handler
专门把异常转换成响应
```
@app.exception_handler(UserNotFoundError)
async def handle_user_not_found(
    request: Request,
    error: UserNotFoundError,
):
    return JSONResponse(
        status_code=404,
        content={
            "code": "USER_NOT_FOUND",
            "message": str(error),
        },
    )
```
适合：
- 业务异常映射
- 请求校验异常
- HTTP 异常
- 统一错误结构

#### Middleware
适合处理请求和响应的通用横切逻辑
```
try:
    response = await call_next(request)
except Exception:
    logger.exception("Unhandled error")
    raise
```

#### 推荐
- Middleware：记录、追踪、通用处理
- Exception Handler：异常到 HTTP 响应的映射

### 面试表达
> FastAPI Middleware 以洋葱模型包裹应用。请求从最外层 Middleware 逐层进入，执行 call_next() 后进入后续应用；响应生成后再从内向外返回。Middleware 适合日志、Request ID、耗时、CORS 和安全响应头等横切能力。具体路由需要的用户、权限和数据库 Session 更适合使用 Dependency，异常到 HTTP 响应的映射则应交给 Exception Handler。